# Test de Conexión con Claude API

Este notebook prueba la conexión con la API de Claude utilizando la clave API desde un archivo .env.

In [12]:
import os
import requests
from dotenv import load_dotenv
from pathlib import Path

# Cargar variables de entorno desde .env en el directorio padre
dotenv_path = Path('../.env')
load_dotenv(dotenv_path=dotenv_path)

# Si no funciona, intenta cargar desde otras ubicaciones comunes
if not os.getenv("ANTHROPIC_API_KEY"):
    # Probar en el directorio actual
    load_dotenv()
    
    # Probar en el directorio raíz del proyecto (2 niveles arriba)
    if not os.getenv("ANTHROPIC_API_KEY"):
        dotenv_path = Path('../../.env')
        load_dotenv(dotenv_path=dotenv_path)

# Verificar que la clave API existe
api_key = os.getenv("ANTHROPIC_API_KEY")
if not api_key:
    raise ValueError("❌ ANTHROPIC_API_KEY no está definida en el archivo .env. Se buscó en el directorio actual, el directorio padre y dos niveles arriba.")
else:
    # Solo mostrar los primeros 5 caracteres por seguridad
    print(f"✓ API_KEY encontrada: {api_key[:5]}...{api_key[-2:]}")

✓ API_KEY encontrada: sk-an...AA


In [7]:
def test_claude_api(auth_method="x-api-key"):
    base_url = "https://api.anthropic.com/v1"
    model = "claude-3-haiku-20240307"
    prompt = "¿Quién eres tú?"
    
    # Configurar encabezados según el método de autenticación
    if auth_method == "x-api-key":
        headers = {
            "x-api-key": api_key,
            "anthropic-version": "2023-06-01",
            "content-type": "application/json"
        }
    else: # "bearer"
        headers = {
            "Authorization": f"Bearer {api_key}",
            "anthropic-version": "2023-06-01",
            "content-type": "application/json"
        }
    
    body = {
        "model": model,
        "max_tokens": 300,
        "temperature": 0.7,
        "messages": [
            {"role": "user", "content": prompt}
        ]
    }
    
    try:
        print(f"Intentando método: {auth_method}")
        response = requests.post(f"{base_url}/messages", headers=headers, json=body)
        
        # Mostrar detalles de la respuesta
        print(f"Código de estado: {response.status_code}")
        if response.status_code != 200:
            print(f"Detalles del error: {response.text}")
            return None
        
        data = response.json()
        return data["content"][0]["text"]
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

In [8]:
# Probar con x-api-key (método oficial según la documentación)
response_xapi = test_claude_api("x-api-key")
if response_xapi:
    print("\n✅ Método x-api-key funcionó correctamente")
    print(f"Respuesta: {response_xapi}")

Intentando método: x-api-key
Código de estado: 200

✅ Método x-api-key funcionó correctamente
Respuesta: Soy un asistente de inteligencia artificial creado por Anthropic. No tengo un cuerpo físico, sino que existo como un programa de software diseñado para ayudar a las personas con una amplia variedad de tareas y preguntas. Mi objetivo es ser útil, informativo y amable en mis interacciones. ¿En qué puedo ayudarte?


In [ ]:
def generate_text_with_claude(prompt, model="claude-3-haiku-20240307", max_tokens=300, temperature=0.7):
    # Usar el método que funcionó en las pruebas anteriores
    # Cambia 'x-api-key' por 'bearer' si ese fue el que funcionó
    auth_method = "x-api-key"  # o "bearer" según la celda que haya funcionado
    
    base_url = "https://api.anthropic.com/v1"
    
    if auth_method == "x-api-key":
        headers = {
            "x-api-key": api_key,
            "anthropic-version": "2023-06-01",
            "content-type": "application/json"
        }
    else: # "bearer"
        headers = {
            "Authorization": f"Bearer {api_key}",
            "anthropic-version": "2023-06-01",
            "content-type": "application/json"
        }
    
    body = {
        "model": model,
        "max_tokens": max_tokens,
        "temperature": temperature,
        "messages": [
            {"role": "user", "content": prompt}
        ]
    }
    
    try:
        response = requests.post(f"{base_url}/messages", headers=headers, json=body)
        response.raise_for_status()
        data = response.json()
        return data["content"][0]["text"]
    except Exception as e:
        print(f"❌ Error: {e}")
        if hasattr(e, 'response') and e.response is not None:
            print(f"Detalles: {e.response.text}")
        return None

In [11]:
# Probar la implementación final
respuesta = generate_text_with_claude("Explica brevemente qué es una API REST")
print(respuesta)

Una API REST (Representational State Transfer) es una interfaz de programación de aplicaciones (API) que sigue los principios de arquitectura REST. Estas son algunas de las características clave de una API REST:

1. Arquitectura cliente-servidor: La API REST separa la lógica del cliente y del servidor, permitiendo que ambos se desarrollen y escalen de forma independiente.

2. Sin estado (stateless): Cada solicitud HTTP contiene toda la información necesaria para ser procesada, sin depender de información almacenada en el servidor.

3. Uso de métodos HTTP: Las API REST utilizan los métodos HTTP (GET, POST, PUT, DELETE, etc.) para realizar las operaciones CRUD (Crear, Leer, Actualizar, Eliminar) sobre los recursos.

4. Recursos identificados por URI: Los recursos de una API REST se identifican mediante Uniform Resource Identifiers (URIs), que permiten acceder a ellos de forma única.

5. Formato de intercambio de datos: Normalmente, las API REST utilizan formatos de datos ligeros y fácile